In [2]:
import torch
import torch.nn as nn
import tiktoken as tt
import math
from torch.nn.functional import softmax
torch.manual_seed(42)

tokenizer = tt.get_encoding("gpt2")

In [11]:
GPT2Config = {
  "activation_function": "gelu_new", # 'new'?
  "architectures": [
    "GPT2LMHeadModel" # anything special bout this?
  ],
  "attn_pdrop": 0.1,
  "bos_token_id": 50256, # same as eos?
  "embd_pdrop": 0.1,
  "eos_token_id": 50256,
  "initializer_range": 0.02,
  "layer_norm_epsilon": 1e-05,
  "model_type": "gpt2",
  "n_ctx": 1024,
  "n_embd": 768,
  "n_head": 12,
  "n_layer": 12,
  "n_positions": 1024,
  "resid_pdrop": 0.1,
  "summary_activation": None,
  "summary_first_dropout": 0.1,
  "summary_proj_to_labels": True,
  "summary_type": "cls_index",
  "summary_use_proj": True,
  "task_specific_params": {
    "text-generation": {
      "do_sample": True,
      "max_length": 50
    }
  },
  "vocab_size": 50257
}

For LLMs we want only the model on the right, the "decoder" <img src="https://www.researchgate.net/publication/338223294/figure/fig2/AS:841443144900609@1577627087767/Transformer-Encoder-Decoder-architecture-taken-from-Vaswani-et-al-9-for-illustration.jpg" alt="drawing" width="400"/>

In [139]:
def scaled_dot_product_attention(queries, keys, values, mask=None):
    """
    - Q, K, V each (n_tokens x out_dim)
    - Q @ K.T => (n_tokens x n_tokens) (similarity score)
    - (Q @ K.T) @ V => (QK^T: n_tokens x n_tokens) x (V: n_tokens x out_dim)
                    => (n_tokens x out_dim) returned (attn)
    - DIM NOT CHANGED by SDP
    """
    similarity_score = queries.matmul(keys.T)
    dk = keys.size(-1)
    denom = torch.sqrt(torch.tensor(dk))
    sdp = 1/denom * similarity_score
    
    if mask is not None:
        sdp += -1e9 * mask # ? 
    
    attn = torch.matmul(softmax(sdp, dim=-1), values)
    return attn

class AttentionHead(nn.Module):
    def __init__(self, in_dim, out_dim): # both equal to embedding_dim
        super().__init__()
        """
        - DIM NOT CHANGED
        - in: n_tokens x embed_dim
        - out: n_tokens x embed_dim
        TODO: batch index
        """
        self.Q_proj = nn.Linear(in_dim, out_dim, bias=False)
        self.K_proj = nn.Linear(in_dim, out_dim, bias=False)
        self.V_proj = nn.Linear(in_dim, out_dim, bias=False)
        
    def forward(self, x): # n_tokens x embed_dim
        Q = self.Q_proj(x) # (x: n_tokens x embed_dim) @ (Q_proj: embed_dim x embed_dim)^T -> (n_tokens x embed_dim)
        K = self.K_proj(x) # dim same as x (")
        V = self.V_proj(x) # dim same as x (")
        return scaled_dot_product_attention(Q, K, V)

class MultiHeadAttention(nn.Module):
    def __init__(self, n_heads, in_dim, out_dim): 
        super().__init__()
        self.head_dim = out_dim // n_heads
        self.attn_heads = nn.ModuleList([
            # each head takes in (n_tokens x head_dim)
            # and returns        (n_tokens x head_dim)
            AttentionHead(in_dim, self.head_dim) for _ in range(n_heads)
        ])

        self.out_proj = nn.Linear(in_dim, out_dim)

    def forward(self, x):
        n_tokens, in_dim = x.shape
        heads = torch.concat([head(x) for head in self.attn_heads], dim=-1)
        # = n_tokens x (head_dim*n_heads = out_dim = embed_dim)
        out = self.out_proj(heads)
        return out

In [142]:
mha = MultiHeadAttention(n_heads=12, in_dim=12, out_dim=12)
print(mha)
mha(torch.ones(3,12))

MultiHeadAttention(
  (attn_heads): ModuleList(
    (0-11): 12 x AttentionHead(
      (Q_proj): Linear(in_features=12, out_features=1, bias=False)
      (K_proj): Linear(in_features=12, out_features=1, bias=False)
      (V_proj): Linear(in_features=12, out_features=1, bias=False)
    )
  )
  (out_proj): Linear(in_features=12, out_features=12, bias=True)
)


tensor([[-0.0340,  0.3106,  0.1165, -0.2825,  0.4577, -0.6174,  0.2731, -0.7435,
         -0.4210,  0.6957,  0.5131,  0.4219],
        [-0.0340,  0.3106,  0.1165, -0.2825,  0.4577, -0.6174,  0.2731, -0.7435,
         -0.4210,  0.6957,  0.5131,  0.4219],
        [-0.0340,  0.3106,  0.1165, -0.2825,  0.4577, -0.6174,  0.2731, -0.7435,
         -0.4210,  0.6957,  0.5131,  0.4219]], grad_fn=<AddmmBackward0>)

In [147]:
class TransformerBlock(nn.Module):
    
    def __init__(self, n_heads, in_dim, out_dim):
        super().__init__()
        self.mha = MultiHeadAttention(n_heads, in_dim, out_dim) # returns in_dim x dk*n_heads
        self.layernorm1 = nn.LayerNorm(GPT2Config['n_embd'], eps=GPT2Config['layer_norm_epsilon'])
        self.layernorm2 = nn.LayerNorm(GPT2Config['n_embd'], eps=GPT2Config['layer_norm_epsilon'])
        self.dropout = nn.Dropout(GPT2Config['attn_pdrop'])
        
        self.ffn = nn.Sequential(
            nn.Linear(in_dim, in_dim * 4),
            nn.GELU(approximate='tanh'),
            nn.Linear(in_dim * 4, in_dim)
        )
        
    def forward(self, x):
        x_old = x
        x = self.layernorm1(x)
        x = self.mha(x)
        x = self.dropout(x)
        x = x + x_old
        
        x_old_2 = x
        x = self.layernorm2(x)
        x = self.ffn(x)
        x = self.dropout(x)
        x = x + x_old_2
        return x

class TransformerDecoder(nn.Module):
    
    def __init__(self, num_layers, num_heads, vocab_size, context_len, embedding_dim):
        super().__init__()
        self.ctx_len = context_len
        self.token_embedding = nn.Embedding(vocab_size, embedding_dim)
        self.positional_embedding = nn.Embedding(context_len, embedding_dim)
        self.dropout = nn.Dropout(0.1)
        
        self.blocks = nn.Sequential(*[
            TransformerBlock(num_heads, embedding_dim, embedding_dim) for _ in range(num_layers) 
        ])
        
        self.linear = nn.Linear(embedding_dim, vocab_size, bias=False)
        
        self.softmax = nn.Softmax(dim=0)
        
    def forward(self, x):
        seq_len = x.shape[-1]
        assert seq_len <= self.ctx_len
        positions = torch.arange(seq_len) # 0, 1, 2, .., seq_len-1
        
        embeddings = self.token_embedding(x) + self.positional_embedding(positions)
        
        x = self.dropout(embeddings)
        x = self.blocks(x)
        x = self.linear(x)
        
        return self.softmax(x)

In [148]:
gpt = TransformerDecoder(
    num_layers=12,
    num_heads=12,
    vocab_size=50257,
    context_len=1024,
    embedding_dim=768
)
gpt

TransformerDecoder(
  (token_embedding): Embedding(50257, 768)
  (positional_embedding): Embedding(1024, 768)
  (dropout): Dropout(p=0.1, inplace=False)
  (blocks): Sequential(
    (0): TransformerBlock(
      (mha): MultiHeadAttention(
        (attn_heads): ModuleList(
          (0-11): 12 x AttentionHead(
            (Q_proj): Linear(in_features=768, out_features=64, bias=False)
            (K_proj): Linear(in_features=768, out_features=64, bias=False)
            (V_proj): Linear(in_features=768, out_features=64, bias=False)
          )
        )
        (out_proj): Linear(in_features=768, out_features=768, bias=True)
      )
      (layernorm1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (layernorm2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (ffn): Sequential(
        (0): Linear(in_features=768, out_features=3072, bias=True)
        (1): GELU(approximate='tanh')
        (2): Linear(in_features=3072, 

In [145]:
sum(p.numel() for p in gpt.parameters())

162989568

In [149]:
tokens = torch.tensor(tokenizer.encode("Hello, you!"))
print(tokens.shape)
gpt(tokens)

torch.Size([4])


tensor([[0.3202, 0.0743, 0.4493,  ..., 0.2298, 0.1017, 0.0544],
        [0.4084, 0.5623, 0.0520,  ..., 0.0751, 0.4643, 0.1287],
        [0.1405, 0.1639, 0.3364,  ..., 0.6248, 0.3968, 0.0826],
        [0.1309, 0.1994, 0.1624,  ..., 0.0703, 0.0372, 0.7343]],
       grad_fn=<SoftmaxBackward0>)

In [159]:
def generate_text_simple(model, idx, max_new_tokens, context_size):
    # idx is (batch, n_tokens) array of indices in the current context
    for _ in range(max_new_tokens):
        
        # Crop current context if it exceeds the supported context size
        # E.g., if LLM supports only 5 tokens, and the context size is 10
        # then only the last 5 tokens are used as context
        idx_cond = idx[-context_size:]
        
        # Get the predictions
        with torch.no_grad():
            logits = model(idx_cond)
        
        # Focus only on the last time step
        # (batch, n_tokens, vocab_size) becomes (batch, vocab_size)
        logits = logits[-1, :]  

        # Apply softmax to get probabilities
        probas = torch.softmax(logits, dim=-1)  # (batch, vocab_size)

        # Get the idx of the vocab entry with the highest probability value
        idx_next = torch.argmax(probas, dim=-1, keepdim=True)  # (batch, 1)
        # Append sampled index to the running sequence
        idx = torch.cat((idx, idx_next), dim=0)  # (batch, n_tokens+1)

    return idx

In [172]:
token_input = torch.tensor(tokenizer.encode("Hello, I"))
res = generate_text_simple(gpt.eval(), token_input, max_new_tokens=6, context_size=1024)
print(res)
print("Generated text:\n\n", tokenizer.decode(res.tolist()))

tensor([15496,    11,   314,  5854,  9701, 23619, 46951, 16920, 28811])
Generated text:

 Hello, I revolution flags ost870 Rocket Archangel


In [135]:
a = torch.randn(10, 3, 12, 5)
b = torch.randn(10, 3, 12, 5)

In [138]:
(a @ b.transpose(2, 3)).shape

torch.Size([10, 3, 12, 12])